In [9]:
import pandas as pd
from tqdm import tqdm
import numpy as np
import os

In [10]:
def id_sort(data):
    data['SortKey'] = data.iloc[:, 0].str.extract(r'(\d+)').astype(int)  # 提取數字部分，轉換為數值型
    data_sorted = data.sort_values(by='SortKey').reset_index(drop=True)  # 按數字部分排序
    data_sorted = data_sorted.drop(columns=['SortKey'])  # 刪除輔助列

    return data_sorted

In [11]:
imputed_data_path = r"V:\dunwei\MACE\dataset\reversed/"
raw_data_path = r"V:\dunwei\MACE\dataset\pre_process/"
signal_data_path = r"V:\dunwei\MACE\dataset\multirocket_result\multirocket\resample_0\MultiRocket_50000\sr2500_500_1000_logistic_merge\combine/"
save_data_path = r"V:\dunwei\MACE\dataset\pre_process\combine_feature/"

In [12]:
if not os.path.exists(save_data_path):
    os.makedirs(save_data_path)
    print(f'create directory: {save_data_path}')
else:
    print(f'{save_data_path} already exists')

V:\dunwei\MACE\dataset\pre_process\combine_feature/ already exists


In [13]:
conversion_id = ['Transformation_ID']
probability = ['prob']
label = ['mace']
feature = ['count_v_ar', 'count_copd', 'smoke_try', 'count_dm', 'male']

In [14]:
imputed_data = pd.read_csv(imputed_data_path + "MACE_imputed_data.csv")
raw_data = pd.read_csv(raw_data_path + "MACE_raw_data.csv")
train_prob = pd.read_csv(signal_data_path + "train_prob.csv").values
test_prob = pd.read_csv(signal_data_path + "test_prob.csv").values
# print(train_prob.shape, test_prob.shape)

In [15]:
mode = 2
'''
mode =1 for missing value
mode =2 for non-missing value(used MIDA)
'''
if mode == 1:
    raw_data = raw_data[conversion_id + feature + label]
    raw_data = id_sort(raw_data)
elif mode == 2:
    imputed_data = imputed_data[conversion_id + feature + label]
    imputed_data = id_sort(imputed_data)



In [16]:
prob_arr = np.vstack([train_prob, test_prob])
prob_df = pd.DataFrame(prob_arr, columns=["ID", "prob"])
prob_sorted = id_sort(prob_df)

In [17]:
prob_count = prob_sorted.iloc[:, 0].value_counts().reset_index()
prob_count.columns = ['ID', 'count']
prob_count_sorted = id_sort(prob_count)
display(prob_count_sorted)

,ID,count
0,MACE001,12549
1,MACE002,12549
2,MACE003,12549
3,MACE004,12549
4,MACE005,12549
...,...,...
516,MACE520,12549
517,MACE521,12549
518,MACE522,12549
519,MACE523,12549


In [18]:
combine = pd.DataFrame()
if mode == 1:
    for i in tqdm(range(len(prob_count_sorted))):
        match_row = raw_data[raw_data.iloc[:, 0] == prob_count_sorted.iloc[i, 0]]
        match_row_duplicate = pd.concat([match_row] * prob_count_sorted.iloc[i, 1], axis=0, ignore_index=True)
        combine = pd.concat([combine, match_row_duplicate], axis=0, ignore_index=True)
elif mode == 2:
    for i in tqdm(range(len(prob_count_sorted))):
        match_row = imputed_data[imputed_data.iloc[:, 0] == prob_count_sorted.iloc[i, 0]]
        match_row_duplicate = pd.concat([match_row] * prob_count_sorted.iloc[i, 1], axis=0, ignore_index=True)
        combine = pd.concat([combine, match_row_duplicate], axis=0, ignore_index=True)

100%|██████████| 521/521 [02:12<00:00,  3.94it/s]


In [ ]:
new_tabular_data = pd.concat([combine, prob_sorted.iloc[:, 1]], axis=1)
new_tabular_data = new_tabular_data[conversion_id + feature + probability + label]
if mode == 1:
    new_tabular_data.to_csv(save_data_path + "combine_tabular_data_missing_value.csv", index=False)
elif mode == 2:
    new_tabular_data.to_csv(save_data_path + "combine_tabular_data_non_missing_value.csv", index=False)

print(f"save path : {save_data_path}")